# 10-naming — Đặt tên cụm bằng Hungarian trên tập test

**Câu hỏi**: bước 10 đặt tên 5 cụm như thế nào, điểm của lớp mới mất bao nhiêu ở bước này (vì mỗi cụm chỉ mang một tên, và vì Hungarian cân theo số ảnh trong khi đại lượng chính cân theo lớp), và cấu trúc cụm của pool có chuyển sang tập test không? (D13, QD15, QD24, QD25; Q10.01, Q10.03)

### Nguồn gốc

Mẫu cell 0 (`lab/README.vi.md` §11): in commit và trạng thái dirty, rồi đặt `XLA_FLAGS` ở chế độ GPU tất định trước mọi lần import jax.

In [1]:
# cell 0: provenance (run first)
import os, subprocess, sys
from pathlib import Path
ROOT = Path.cwd().resolve()
while not (ROOT / "main.py").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
os.chdir(ROOT); sys.path[:0] = [str(ROOT), str(ROOT / "lab")]
import replay; os.environ["XLA_FLAGS"] = replay._xla_flags()   # GPU deterministic mode before any jax import
git = lambda *a: subprocess.run(["git", *a], capture_output=True, text=True).stdout.strip()
print("commit", git("rev-parse", "--short", "HEAD"),
      "| code dirty", bool(git("status", "--porcelain", "--", ".", ":(exclude)runs", ":(exclude)notes",
                               ":(exclude)lab/*.ipynb", ":(exclude)lab/*.md")),
      "| XLA_FLAGS", os.environ["XLA_FLAGS"])

commit 00a17c4 | code dirty False | XLA_FLAGS --xla_gpu_deterministic_ops=true


- **Kết quả**: commit `00a17c4`, code dirty False, XLA_FLAGS `--xla_gpu_deterministic_ops=true`.

## 1. Kiểm

### 1.1 Chạy lại bước 10 ở r0 s4

Chạy lại bước 10 của `gc10_v2_ft_r0_s4` từ ảnh chụp 09 bằng `replay.rerun`, rồi so hai khoá `pi_test` và `lut` với ảnh chụp 10. Đây là hai khoá rời rạc, nên phải trùng tuyệt đối. In thêm `pi_test` theo tên lớp. Lớp mới đánh số 0–4 theo thứ tự trong file split, sau khi trừ `offset` = 5.

Dự đoán [SUY LUẬN]:
- `rerun` OK 2/2 khoá, vì bước 10 chỉ gồm `predict_raw` và Hungarian, chạy trên CPU và không rút RNG;
- ở r0 s4, cụm 1 (cụm do waist sở hữu trên pool, `09d` §2.1) mang tên waist, cụm 0 (cụm do inclusion sở hữu) mang tên inclusion, một trong ba cụm silk mang tên silk; hai cụm silk còn lại bị gán tên rolled và crease.

In [2]:
# 1.1 rerun step 10 from the step-09 snapshot; compare pi_test and lut with snapshot 10
import json
import numpy as np

RUN = "gc10_v2_ft_r0_s4"
out, ok = replay.rerun(RUN, "10")

split = json.load(open(ROOT / "splits" / "gc10_split.json"))
novel = [c.split("_")[0] for c in split["classes"][split["n_base"]:]]   # novel ids 0..4 after offset
print("novel ids:", dict(enumerate(novel)))
print("pi_test (cluster -> name):", {k: novel[v] for k, v in sorted(out["pi_test"].items())})
print("lut:", out["lut"].tolist())

/home/rito/miniforge3/envs/vbcgcd/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


rerun step 10 of gc10_v2_ft_r0_s4 | device gpu | GPU now NVIDIA GeForce RTX 4060 Laptop GPU, 621.01 | GPU of the run NVIDIA GeForce RTX 4060 Laptop GPU, 617.42 | driver differs: float outputs may move in the last bits
hungarian stage 1: mode=eval pi_test={2: 0, 0: 1, 3: 2, 4: 3, 1: 4}
  OK   pi_test      dict                   max|diff| nan
  OK   lut          array int64            max|diff| 0
rerun step 10: OK (2/2 keys)
novel ids: {0: 'silk', 1: 'inclusion', 2: 'rolled', 3: 'crease', 4: 'waist'}
pi_test (cluster -> name): {0: 'inclusion', 1: 'waist', 2: 'silk', 3: 'rolled', 4: 'crease'}
lut: [0, 1, 2, 3, 4, 7, 5, 8, 9, 6]


- **Kết quả**:
  - `rerun` bước 10 ở r0 s4: OK 2/2 khoá (`pi_test`, `lut` trùng tuyệt đối). Driver GPU đổi từ 617.42 sang 621.01, nhưng bước 10 không có đầu ra số thực nên lời cảnh báo không ảnh hưởng.
  - Tên cụm: cụm 0 → inclusion, 1 → waist, 2 → silk, 3 → rolled, 4 → crease. `lut` = [0, 1, 2, 3, 4, 7, 5, 8, 9, 6].
- **Ý nghĩa**:
  - Đúng như dự đoán. Cụm silk lớn nhất (cụm 2: 169 silk, 76 inclusion trên pool, `09d` §2.1) mang tên silk.
  - Hai cụm gần như thuần silk (cụm 3 có 118/121 ảnh silk, cụm 4 có 103/103) bị gán tên rolled và crease. Ở r0 s4 vì thế không cụm nào thực sự chứa rolled hay crease mà vẫn mang tên của chúng. Đây là hiện tượng "một cụm một tên" của Q10.01.

### 1.2 Tên cụm trên 15 ảnh chụp, so với lớp sở hữu cụm trong pool

Với mỗi ảnh chụp, đọc thẳng các file ảnh chụp (không chạy lại bước nào):
- tính lại `pi_test = hungarian_map(cluster.predict_raw(test1_x), true_test1_y − offset, 5)` và so với `pi_test` trong ảnh chụp 10;
- in tên của từng cụm cạnh lớp sở hữu cụm đó trong pool (lớp đông nhất, tính cả lớp cũ, như `score09`);
- đếm số cụm có tên khác lớp sở hữu.

Dự đoán [SUY LUẬN]:
- trùng 15/15;
- số cụm có tên khác lớp sở hữu = số cụm silk thừa: 2 ở 12 ảnh chụp mà silk sở hữu 3 cụm, 1 ở r0 s0, r0 s1, r2 s0 (silk sở hữu 2 cụm, `09d` §1.2); cụm thừa nhận các tên rolled, crease hoặc inclusion;
- ở ảnh chụp có cụm crease (r0 s0–s2, r1 s0–s2), cụm đó mang tên crease.

In [3]:
# 1.2 pi_test recomputed on 15 snapshots vs snapshot 10; cluster name vs pool owner
import pandas as pd
from clustering.clustering_base import hungarian_map

names = [c.split("_")[0] for c in split["classes"]]                 # 10 classes, true ids 0..9
RUNS = [f"gc10_v2_ft_r{r}_s{s}" for r in range(3) for s in range(5)]
rows, NAMING = [], {}
for run in RUNS:
    b = replay.snapshots(run)
    o4, o9, o10 = b["04"]["out"], b["09"]["out"], b["10"]["out"]
    off = o4["offset"]
    test_ids = o9["cluster"].predict_raw(o4["test1_x"])                 # (214,) cluster id of each novel test image
    pi = hungarian_map(test_ids, np.asarray(o4["true_test1_y"]).astype(int) - off, 5)
    y, ids = np.asarray(o4["true_pool_y"]).astype(int), np.asarray(o9["ids"])
    n = np.zeros((10, 5), int); np.add.at(n, (y, ids), 1)              # pool: class x cluster counts
    owner = [names[i] for i in n.argmax(0)]
    named = [names[off + pi[k]] for k in range(5)]
    NAMING[run] = dict(pi=pi, test_ids=test_ids, owner=owner)
    rows.append(dict(run=run.replace("gc10_v2_ft_", ""), same_as_snapshot=pi == o10["pi_test"],
                     named=" ".join(f"{k}:{named[k]}" for k in range(5)),
                     owner=" ".join(f"{k}:{owner[k]}" for k in range(5)),
                     n_differ=sum(a != b for a, b in zip(named, owner))))
T12 = pd.DataFrame(rows).set_index("run")
pd.set_option("display.width", 200)
print(T12.to_string())
print("same as snapshot:", int(T12["same_as_snapshot"].sum()), "/ 15")

       same_as_snapshot                                         named                                          owner  n_differ
run                                                                                                                           
r0_s0              True  0:rolled 1:crease 2:waist 3:silk 4:inclusion     0:silk 1:crease 2:waist 3:silk 4:inclusion         1
r0_s1              True  0:inclusion 1:silk 2:rolled 3:crease 4:waist     0:inclusion 1:silk 2:silk 3:crease 4:waist         1
r0_s2              True  0:rolled 1:inclusion 2:crease 3:waist 4:silk          0:silk 1:silk 2:crease 3:waist 4:silk         2
r0_s3              True  0:waist 1:crease 2:silk 3:rolled 4:inclusion       0:waist 1:silk 2:silk 3:silk 4:inclusion         2
r0_s4              True  0:inclusion 1:waist 2:silk 3:rolled 4:crease       0:inclusion 1:waist 2:silk 3:silk 4:silk         2
r1_s0              True  0:crease 1:silk 2:inclusion 3:rolled 4:waist          0:silk 1:silk 2:crease 3:silk 4:

- **Kết quả**:
  - `pi_test` tính lại trùng ảnh chụp 10 ở 15/15.
  - Số cụm có tên khác lớp sở hữu trong pool: 1–4. Dự đoán "bằng số cụm silk thừa" đúng ở 9/15 ảnh chụp (cả 5 ảnh chụp của r0; r1 s3; r2 s1, s2, s4), sai ở 6/15:
    - r1 s0, s1, s2: cụm do crease sở hữu trong pool mang tên inclusion; tên crease rơi vào một cụm silk;
    - r1 s4, r2 s3: tên silk và inclusion đổi chỗ giữa cụm do inclusion sở hữu và một cụm silk;
    - r2 s0 (silk 2 cụm, inclusion 2 cụm): một cụm inclusion mang tên silk, hai cụm silk mang tên rolled và crease.
  - Cụm crease mang tên crease ở r0 s0–s2 (3/3); ở r1 s0–s2 thì không (0/3).
- **Ý nghĩa**:
  - Dự đoán chỉ đúng ở r0. Tên cụm không suy ra được từ lớp sở hữu cụm trong pool: ở 6/15 ảnh chụp, Hungarian trên tập test đặt tên ngược với pool, kể cả ở cụm do chính crease sở hữu.
  - [SUY LUẬN] Có hai nguồn có thể: (i) trên tập test, ảnh lớp mới rơi vào cụm khác với pool (Q10.03); (ii) Hungarian cân theo số ảnh, nên 43 ảnh inclusion thắng 11 ảnh crease khi tranh cùng một cụm. Mục 3 tách hai nguồn này bằng cách đặt tên tối ưu cho balanced; mục 5 đo nguồn (i).

## 2. Code

### 2.1 Bước 10 tối ưu bảng nào

In code của `s10_naming` và `hungarian_map`, rồi in đúng bảng mà Hungarian tối ưu ở r0 s4: số ảnh test lớp mới theo (lớp thật × cụm), cụm lấy từ `predict_raw` của k-means. Đánh dấu ô được chọn, in tổng số ảnh đúng dưới cách đặt tên của pipeline. Bảng pool (cụm × lớp) để so sánh nằm ở `09d` §2.1.

**Cách làm**: Hungarian (Kuhn 1955) chọn song ánh π: cụm → lớp sao cho Σ_k n_{π(k),k} lớn nhất, với n_{c,k} là số ảnh test của lớp c rơi vào cụm k. Mỗi cụm một tên, mỗi tên một cụm.

Dự đoán [SUY LUẬN]: ở r0 s4, gần như toàn bộ 11 ảnh crease và 6 ảnh rolled rơi vào cụm 1 (cụm mang tên waist), nên recall mức cụm của crease và rolled gần 0. Tổng số ảnh đúng khoảng 120–150/214.

In [4]:
# 2.1 the code of step 10 and the table Hungarian maximises at r0 s4
import inspect
from clustering import clustering_base
print(inspect.getsource(replay.s10_naming))
print(inspect.getsource(clustering_base.hungarian_map))

nov = names[5:]
t = NAMING[RUN]
yt = np.asarray(replay.snapshots(RUN)["04"]["out"]["true_test1_y"]).astype(int) - 5   # novel ids 0..4
m = np.zeros((5, 5), int); np.add.at(m, (yt, t["test_ids"]), 1)                     # rows true class, cols cluster
tab = pd.DataFrame(m, index=nov, columns=[f"c{k}:{nov[t['pi'][k]]}" for k in range(5)])
tab["n_test"] = m.sum(axis=1)
print(tab.to_string())
picked = sum(m[t["pi"][k], k] for k in range(5))
print(f"correct under the pipeline naming: {picked}/{m.sum()} | per class:",
      {nov[t['pi'][k]]: f"{m[t['pi'][k], k]}/{m[t['pi'][k]].sum()}" for k in range(5)})

def s10_naming(st):
    """10 Cluster names for evaluation only (D13): Hungarian on the novel test set.
    In: cluster, test1_x, true_test1_y, offset. Out: pi_test (cluster -> novel id - offset), lut (true id -> cluster space)."""
    from clustering.clustering_base import hungarian_map
    a, off = st["args"], st["offset"]
    pi = hungarian_map(st["cluster"].predict_raw(st["test1_x"]), np.asarray(st["true_test1_y"]).astype(int) - off,
                       a.increment)
    print(f"hungarian stage 1: mode=eval pi_test={pi}")
    lut = np.arange(a.num_classes)
    for c, t in pi.items():
        lut[off + t] = off + c
    return dict(pi_test=pi, lut=lut)

def hungarian_map(cluster_ids, true_ids, k):
    '''
    Best one-to-one map cluster id -> true id, both in 0..k-1 (Hungarian on the confusion matrix).
    Called at EVALUATION time with test labels (D13 fix); never on unlabeled training data.
    '''
    cm = confusion_matrix(y_true=true_ids, y_pred=cluster_ids, labels=np.arange(k)

- **Kết quả**:
  - `s10_naming` gọi `hungarian_map` một lần trên bảng (5 lớp mới × 5 cụm) của 214 ảnh test, với cụm lấy từ `predict_raw` của k-means. `lut` chỉ dùng để đổi nhãn test sang không gian cụm.
  - r0 s4, theo cách đặt tên của pipeline: đúng 129/214 ảnh. silk 86/124, inclusion 15/43, waist 28/30, rolled 0/6, crease 0/11. Balanced mức cụm (0.694 + 0.349 + 0.933 + 0 + 0)/5 = 0.395.
  - Cả 6 ảnh rolled và 7/11 ảnh crease rơi vào cụm c1 (tên waist); 4 ảnh crease rơi vào c0 (tên inclusion).
  - Hai cụm gần như thuần silk mang tên rolled (c3) và crease (c4) chỉ nhận 21 và 9 ảnh silk trên test, tức 30/124 = 24% số ảnh silk. Trên pool, hai cụm này chứa 221/420 = 53% số ảnh silk (`09d` §2.1).
- **Ý nghĩa**:
  - Dự đoán đúng ở rolled (6/6 trong cụm waist) và ở tổng 129/214; đúng một phần ở crease (7/11, không phải gần hết).
  - Hai tên rolled và crease đặt lên hai cụm silk đều cho 0 ảnh đúng; chúng mất toàn bộ vì một cụm chỉ mang một tên (Q10.01).
  - [SUY LUẬN] Hai cụm silk c3, c4 co từ 53% số ảnh silk trên pool xuống 24% trên test: một dấu hiệu sớm cho Q10.03 (cụm bám cuộn không chuyển sang test), đo đủ ở mục 5.

## 3. Ba cách đặt tên

### 3.1 Balanced novel mức cụm dưới ba cách đặt tên, trên 15 ảnh chụp

Với mỗi ảnh chụp, từ bảng n_{c,k} (5 lớp mới × 5 cụm) của 214 ảnh test, như 2.1:
- (a) tên của pipeline: song ánh làm cực đại Σ_k n_{π(k),k} (QD15);
- (b) song ánh làm cực đại Σ_k n_{π(k),k}/n_{π(k)}, tức Hungarian trên ma trận recall: tối ưu cho chính balanced novel (QD25);
- (c) nhiều–một: mỗi cụm mang tên lớp sở hữu nó trong pool (như 1.2); cụm do lớp cũ sở hữu không mang tên lớp mới nào. Chỉ dùng để chẩn đoán, không phải cách đặt tên dùng được.

Với mỗi cách, tính balanced novel mức cụm (trung bình recall của 5 lớp) và số ảnh đúng; in thêm các cụm mà (b) đặt tên khác (a). Trung bình theo FT seed.

**Cách làm**: (c) − (b) là phần mất vì mỗi cụm chỉ mang một tên, trên chính phân hoạch K = 5; (b) − (a) là phần mất vì Hungarian cân theo số ảnh trong khi đại lượng chính cân theo lớp.

Dự đoán [SUY LUẬN]:
- (b) ≥ (a) ở mọi ảnh chụp (đúng theo cách dựng); (b) khác (a) ở khoảng 5–10/15 ảnh chụp; khoảng cách trung vị 1–5 điểm;
- (b) có thể đạt điểm bằng cách bỏ một lớp lớn để lấy một lớp rất nhỏ. Ví dụ ở r0 s4, (b) đặt c1 là rolled thay vì waist, vì 6/6 ảnh rolled (recall 1.0) nặng hơn 28/30 ảnh waist (0.933). Nếu điều này lặp lại nhiều, (b) là thước đo nhiễu hơn chứ chưa chắc tốt hơn;
- (c) cao hơn (a) khoảng 3–8 điểm, chủ yếu vì silk lấy lại các cụm silk thừa.

In [5]:
# 3.1 cluster-level balanced novel on the test set under three namings, 15 snapshots
from scipy.optimize import linear_sum_assignment

def table_test(run):
    """n[c, k]: novel test images of class c (0..4) in cluster k (k-means predict_raw)."""
    t = NAMING[run]
    yt = np.asarray(replay.snapshots(run)["04"]["out"]["true_test1_y"]).astype(int) - 5
    n = np.zeros((5, 5), int); np.add.at(n, (yt, t["test_ids"]), 1)
    return n

def score_naming(n, name_of):
    """name_of[k] = novel class of cluster k, or -1 (no novel name). Returns balanced novel, images correct."""
    hit = np.zeros(5)
    for k, c in enumerate(name_of):
        if c >= 0:
            hit[c] += n[c, k]
    return float((hit / n.sum(axis=1)).mean()), int(hit.sum())

rows3 = []
for run in RUNS:
    n, t = table_test(run), NAMING[run]
    a = [t["pi"][k] for k in range(5)]                                   # (a) pipeline, count-optimal
    r_, c_ = linear_sum_assignment(-(n / n.sum(axis=1, keepdims=True)))  # (b) recall-optimal; rows = classes
    b = [None] * 5
    for c, k in zip(r_, c_):
        b[k] = int(c)
    own = [nov.index(o) if o in nov else -1 for o in t["owner"]]         # (c) many-to-one by pool owner
    (ba, ca), (bb, cb), (bc, cc) = (score_naming(n, x) for x in (a, b, own))
    rows3.append(dict(run=run.replace("gc10_v2_ft_", ""), bal_a=ba, bal_b=bb, bal_c=bc, cor_a=ca, cor_b=cb, cor_c=cc,
                      b_renames=" ".join(f"c{k}:{nov[a[k]]}->{nov[b[k]]}" for k in range(5) if a[k] != b[k]) or "-"))
T31 = pd.DataFrame(rows3).set_index("run")
print(T31.round(3).to_string())
T31["ft"] = [i[:2] for i in T31.index]
print("\nmean by FT seed:")
print(T31.groupby("ft")[["bal_a", "bal_b", "bal_c", "cor_a", "cor_b", "cor_c"]].mean().round(3).to_string())
print("b differs from a in", int((T31["b_renames"] != "-").sum()), "/ 15 snapshots")

       bal_a  bal_b  bal_c  cor_a  cor_b  cor_c                                                                                  b_renames
run                                                                                                                                       
r0_s0  0.503  0.562  0.554    131     86    163                  c0:rolled->silk c1:crease->rolled c3:silk->inclusion c4:inclusion->crease
r0_s1  0.498  0.553  0.550    130     84    162                  c0:inclusion->crease c1:silk->inclusion c2:rolled->silk c3:crease->rolled
r0_s2  0.503  0.558  0.511    121    119    152                                                        c0:rolled->crease c2:crease->rolled
r0_s3  0.390  0.411  0.439    128     96    158                c0:waist->rolled c1:crease->waist c3:rolled->inclusion c4:inclusion->crease
r0_s4  0.395  0.411  0.444    129     96    159                c0:inclusion->crease c1:waist->rolled c3:rolled->inclusion c4:crease->waist
r1_s0  0.398  0.521  0.551 

- **Kết quả** (balanced novel mức cụm; số ảnh đúng trên 214):
  - Trung bình theo FT seed, (a) / (b) / (c): r0 0.458 / 0.499 / 0.499; r1 0.386 / 0.483 / 0.497; r2 0.405 / 0.423 / 0.452.
  - (b) − (a): r0 +4.1, r1 +9.7, r2 +1.8 điểm. Trung vị trên 15 ảnh chụp +3.3 điểm (từ 0 đến +13.0).
  - (b) đặt tên khác (a) ở 15/15 ảnh chụp, nhưng ở r2 s2 và r2 s4 chỉ là đổi rolled ↔ crease giữa hai cụm đều có recall 0, nên balanced bằng nhau. Khác thật ở 13/15.
  - (b) giữ ít ảnh đúng hơn (a) 0–40 ảnh, chủ yếu mất ảnh silk: r0 96.2 so với 127.8.
  - (b) lấy cụm của waist đưa cho rolled ở 4/15 ảnh chụp: r0 s3, r0 s4, r1 s3, r2 s1.
  - Ở r1 s0–s2, (b) trả tên crease về cho cụm do crease sở hữu trong pool (c2), vốn bị (a) đặt tên inclusion (1.2).
  - (c) − (b): r0 0, r1 +1.4, r2 +2.9 điểm.
- **Ý nghĩa**:
  - Dự đoán đúng ở khoảng cách trung vị (1–5 điểm) và ở (c) − (a) cho r0, r2 (+4.1, +4.7). Sai ở số ảnh chụp mà (b) khác (a) (13/15 so với 5–10), và ở r1, nơi (c) − (a) là +11.1 điểm.
  - Phần lớn khoảng (c) − (a) đến từ việc Hungarian cân theo số ảnh, không từ ràng buộc một tên một cụm.
  - [SUY LUẬN] Ở mức cụm, khoảng cách r0 − r1 là 7.2 điểm theo (a) nhưng chỉ 1.6 theo (b). Như vậy cách đặt tên theo số ảnh có thể giải thích phần lớn chỗ r1 thua r0 ở tầng 2 dù bằng nhau ở tầng 1 (Q14.04). Phải kiểm lại ở 14 bằng dự đoán của bộ phân lớp.
  - Giới hạn: (b) được chọn trên chính 214 ảnh mà nó chấm, trong số 120 song ánh, với lớp chỉ 6 và 11 ảnh. Vì vậy (b) có thể bị thổi phồng; 3.2 đo phần này. Việc (b) đổi cụm waist lấy 6 ảnh rolled ở 4/15 ảnh chụp cho thấy nó nhạy với vài ảnh.

### 3.2 Phần thổi phồng do chọn và chấm trên cùng tập

Chia đôi 214 ảnh test theo từng lớp (giữ tỉ lệ lớp), 500 lần ngẫu nhiên (`RandomState(10)`, cố định). Mỗi lần: chọn cách đặt tên (a) và (b) trên một nửa, chấm balanced novel mức cụm trên nửa còn lại, rồi đổi vai hai nửa. So với (a), (b) chấm ngay trên tập đã dùng để chọn (3.1).

**Cách làm**: phép kiểm chéo hai nửa (2-fold cross-validation, Stone 1974) tách việc chọn khỏi việc chấm. Hiệu (chấm trên cùng tập) − (chấm chéo) là phần thổi phồng do chọn trên chính tập chấm (optimism; Efron 1983).

Dự đoán [SUY LUẬN]:
- (b) bị thổi phồng nhiều hơn (a), khoảng 1–4 điểm, vì trọng số 1/n_c phóng to các lớp 6 và 11 ảnh;
- sau khi kiểm chéo, (b) vẫn hơn (a) ở r1 ít nhất 5 điểm; ở r0 và r2, khoảng cách còn 0–3 điểm.

In [6]:
# 3.2 split-half cross-fitting: choose naming (a) or (b) on one half of the test set, score on the other half
def naming_count(n):
    r_, c_ = linear_sum_assignment(-n); out = [None] * 5
    for c, k in zip(r_, c_): out[k] = int(c)
    return out

def naming_recall(n):
    return naming_count(n / np.maximum(n.sum(axis=1, keepdims=True), 1))

rs = np.random.RandomState(10)
rows32 = []
for run in RUNS:
    t = NAMING[run]
    yt = np.asarray(replay.snapshots(run)["04"]["out"]["true_test1_y"]).astype(int) - 5
    ids = np.asarray(t["test_ids"])
    cv = {"a": [], "b": []}
    for _ in range(500):
        half = np.zeros(len(yt), bool)
        for c in range(5):                                                # stratified split
            idx = rs.permutation(np.flatnonzero(yt == c)); half[idx[: len(idx) // 2]] = True
        for fit, ev in ((half, ~half), (~half, half)):
            nf = np.zeros((5, 5), int); np.add.at(nf, (yt[fit], ids[fit]), 1)
            ne = np.zeros((5, 5), int); np.add.at(ne, (yt[ev], ids[ev]), 1)
            cv["a"].append(score_naming(ne, naming_count(nf))[0])
            cv["b"].append(score_naming(ne, naming_recall(nf))[0])
    rows32.append(dict(run=run.replace("gc10_v2_ft_", ""), a_same=T31.loc[run.replace("gc10_v2_ft_", ""), "bal_a"],
                       b_same=T31.loc[run.replace("gc10_v2_ft_", ""), "bal_b"],
                       a_cv=np.mean(cv["a"]), b_cv=np.mean(cv["b"])))
T32 = pd.DataFrame(rows32).set_index("run")
T32["opt_a"], T32["opt_b"], T32["b_minus_a_cv"] = T32.a_same - T32.a_cv, T32.b_same - T32.b_cv, T32.b_cv - T32.a_cv
print(T32.round(3).to_string())
T32["ft"] = [i[:2] for i in T32.index]
print("\nmean by FT seed:")
print(T32.groupby("ft").mean().round(3).to_string())

       a_same  b_same   a_cv   b_cv  opt_a  opt_b  b_minus_a_cv
run                                                            
r0_s0   0.503   0.562  0.508  0.541 -0.005  0.020         0.033
r0_s1   0.498   0.553  0.492  0.533  0.006  0.019         0.041
r0_s2   0.503   0.558  0.503  0.545 -0.000  0.013         0.041
r0_s3   0.390   0.411  0.390  0.388  0.001  0.024        -0.002
r0_s4   0.395   0.411  0.395  0.389  0.000  0.022        -0.006
r1_s0   0.398   0.521  0.394  0.485  0.004  0.036         0.091
r1_s1   0.394   0.524  0.399  0.494 -0.005  0.030         0.095
r1_s2   0.398   0.521  0.403  0.482 -0.005  0.039         0.079
r1_s3   0.369   0.476  0.373  0.472 -0.004  0.004         0.099
r1_s4   0.368   0.375  0.366  0.339  0.002  0.036        -0.028
r2_s0   0.392   0.416  0.383  0.355  0.009  0.061        -0.028
r2_s1   0.408   0.440  0.408  0.411  0.000  0.029         0.003
r2_s2   0.442   0.442  0.442  0.409  0.000  0.033        -0.033
r2_s3   0.346   0.379  0.345  0.321  0.0

- **Kết quả** (500 lần chia đôi theo lớp; trung bình theo FT seed r0 / r1 / r2):
  - Phần thổi phồng: (a) 0.000 / −0.002 / 0.002, tức gần 0; (b) 0.020 / 0.029 / 0.044 (từng ảnh chụp: 0.004–0.061).
  - Sau khi kiểm chéo, (b) − (a): r0 +2.2, r1 +6.7, r2 −2.4 điểm. Trung bình 3 FT seed +2.2.
  - Ở r1, (b) hơn (a) 7.9–9.9 điểm tại s0–s3; riêng r1 s4 kém 2.8. Ở r2, (b) kém (a) tại 4/5 ảnh chụp.
- **Ý nghĩa**:
  - Dự đoán đúng ở độ thổi phồng của (b) (1–4 điểm; r2 cao hơn một chút, 4.4) và ở r1 (≥ 5 điểm). Sai ở r2: sau kiểm chéo, (b) thua (a).
  - Cách đặt tên theo số ảnh (a) ổn định khi đổi nửa test. (b) nhạy với vài ảnh của các lớp 6 và 11 ảnh, nên không phải định nghĩa tốt hơn cho đại lượng chính. Giữ (a) (QD15, QD25); không đổi quyết định nào.
  - Phần thật của 3.1 nằm ở r1: tên crease bị đặt cho cụm khác do cân theo số ảnh, và phần này vẫn còn sau kiểm chéo (+6.7 điểm).
  - [SUY LUẬN] Đây là một ứng viên cụ thể cho Q14.04 (r1 bằng r0 ở tầng 1 nhưng kém ở tầng 2), cần xác nhận ở 14 bằng dự đoán của bộ phân lớp.

## 4. Sổ mất điểm

### 4.1 Cột 09 và 10 trên 15 ảnh chụp

Gọi `score.ledger` cho 15 ảnh chụp (QD24, thiết kế ở dòng Sửa của QD24). Kiểm: balanced của cột 10 trên `test1` phải bằng (a), và của cột 09 phải bằng (c) ở 3.1, ở 15/15, vì cùng định nghĩa nhưng code khác. In `ledger_summary` của r0 s4 cho cả `test1` và `pool`.

Dự đoán [SUY LUẬN]:
- trùng 15/15;
- r0 s4, `test1`: silk mất 30 ảnh từ 09 sang 10 (21 + 9 ảnh trong hai cụm silk mang tên rolled, crease; 2.1), không lớp nào được thêm ảnh; inclusion và waist giữ nguyên;
- r0 s4, `pool`: silk mất 221 ảnh (118 + 103 trong cụm 3, 4).

In [7]:
# 4.1 loss ledger, columns 09 and 10, 15 snapshots; check against 3.1; r0 s4 summaries
import score

LEDGER = {run: score.ledger(run) for run in RUNS}
chk = []
for run in RUNS:
    s = score.ledger_summary(LEDGER[run]["test1"], names, ["09", "10"]).loc["balanced"]
    key = run.replace("gc10_v2_ft_", "")
    chk.append(dict(run=key, b09=s["09"], c_31=T31.loc[key, "bal_c"], b10=s["10"], a_31=T31.loc[key, "bal_a"]))
chk = pd.DataFrame(chk).set_index("run")
ok = np.allclose(chk.b09, chk.c_31) and np.allclose(chk.b10, chk.a_31)
print("matches 3.1 (09 = c, 10 = a):", ok, f"({int((np.isclose(chk.b09, chk.c_31) & np.isclose(chk.b10, chk.a_31)).sum())}/15)")
for st in ("test1", "pool"):
    print(f"\nr0 s4, {st}:")
    print(score.ledger_summary(LEDGER[RUN][st], names, ["09", "10"]).round(3).to_string())

matches 3.1 (09 = c, 10 = a): True (15/15)

r0 s4, test1:
               n       09      10  lost_09_10  gained_09_10
silk       124.0  116.000  86.000        30.0           0.0
inclusion   43.0   15.000  15.000         0.0           0.0
rolled       6.0    0.000   0.000         0.0           0.0
crease      11.0    0.000   0.000         0.0           0.0
waist       30.0   28.000  28.000         0.0           0.0
balanced     NaN    0.444   0.395         NaN           NaN

r0 s4, pool:
               n       09       10  lost_09_10  gained_09_10
silk       420.0  390.000  169.000       221.0           0.0
inclusion  135.0   46.000   46.000         0.0           0.0
rolled      17.0    0.000    0.000         0.0           0.0
crease      32.0    0.000    0.000         0.0           0.0
waist       94.0   79.000   79.000         0.0           0.0
balanced     NaN    0.422    0.317         NaN           NaN


- **Kết quả**:
  - Balanced của cột 09 trùng (c) và của cột 10 trùng (a) ở 3.1, 15/15 ảnh chụp (code `3aeb5ae`).
  - r0 s4, `test1`: từ 09 sang 10, silk mất 30 ảnh (116 → 86), không lớp nào được thêm ảnh; inclusion 15, waist 28, rolled 0, crease 0 ở cả hai cột. Balanced 0.444 → 0.395.
  - r0 s4, `pool`: silk mất 221 ảnh (390 → 169), các lớp khác không đổi. Balanced 0.422 → 0.317; số 0.317 trùng balanced tầng 1 của r0 s4 (`09d` §1.2), tức ở ảnh chụp này tên đặt trên test trùng song ánh tối ưu trên pool.
- **Ý nghĩa**:
  - Đúng như dự đoán ở cả ba ý. Ở r0 s4, toàn bộ phần mất từ 09 sang 10 là ảnh silk trong hai cụm silk thừa; rolled và crease đã mất từ 09, vì không có cụm nào thuộc về chúng.
  - Silk mất 53% ảnh pool (221/420) nhưng chỉ 24% ảnh test (30/124) ở bước này: hai cụm silk thừa nhận ít ảnh test hơn hẳn. Mục 5 đo điều này trên 15 ảnh chụp.

## 5. Cụm của pool có chuyển sang tập test không (Q10.03)

### 5.1 Trong từng lớp: phân bố qua 5 cụm, pool so với test

Với mỗi ảnh chụp và mỗi lớp mới c:
- p_c = phân bố ảnh pool của lớp c qua 5 cụm (`ids`); q_c = phân bố ảnh test của lớp c (`predict_raw`, cột `cluster` của sổ mất điểm);
- khoảng cách tổng biến phân (total variation) TV = ½ Σ_k |p_c(k) − q_c(k)|;
- mức nền: gộp ảnh pool và ảnh test của lớp c, hoán vị nhãn "pool/test" giữ nguyên cỡ hai phía, 1000 lần (`RandomState(51)`); p = (1 + số lần TV_hoán vị ≥ TV) / 1001.

So trong từng lớp nên khác biệt về thành phần lớp giữa pool và test không lẫn vào. Báo trung vị TV và số ảnh chụp có p < 0.05 trên 5 ảnh chụp của mỗi FT seed.

**Giới hạn**: tâm cụm học từ chính ảnh pool, ảnh test thì không. Vì vậy pool và test không hoàn toàn hoán đổi được cho nhau, và một phần TV có thể chỉ do chênh lệch này, không do cuộn. Đối chứng: waist (cuộn không chia cụm, `09d` §6.1) phải có TV nhỏ; mục 5.2 nối TV với mức tập trung theo cuộn.

Dự đoán [SUY LUẬN]:
- silk: TV 0.2–0.35, p < 0.05 ở ít nhất 4/5 ảnh chụp mỗi FT seed (r0 s4 tính tay từ 2.1 và `09d` §2.1: khoảng 0.32);
- inclusion: TV vượt nền ở phần lớn ảnh chụp;
- waist: TV < 0.1, hiếm khi vượt nền; rolled và crease: TV dao động lớn vì chỉ 6 và 11 ảnh test, hiếm khi vượt nền.

In [8]:
# 5.1 per novel class: distribution over clusters, pool vs test; total variation with a within-class permutation null
rs = np.random.RandomState(51)
rows51 = []
for run in RUNS:
    L = LEDGER[run]
    for c in range(5, 10):
        cp = L["pool"].loc[L["pool"].y == c, "cluster"].to_numpy()
        ct = L["test1"].loc[L["test1"].y == c, "cluster"].to_numpy()
        tv = lambda a, b: 0.5 * np.abs(np.bincount(a, minlength=5) / len(a) - np.bincount(b, minlength=5) / len(b)).sum()
        obs, allc = tv(cp, ct), np.r_[cp, ct]
        null = np.empty(1000)
        for i in range(1000):
            q = rs.permutation(allc); null[i] = tv(q[: len(cp)], q[len(cp):])
        rows51.append(dict(run=run.replace("gc10_v2_ft_", ""), ft=run[11:13], cls=names[c], n_pool=len(cp), n_test=len(ct),
                           tv=obs, null_mean=null.mean(), p=(1 + (null >= obs).sum()) / 1001))
T51 = pd.DataFrame(rows51)
print(T51[T51.run == "r0_s4"].round(3).to_string(index=False))
summ = T51.groupby(["cls", "ft"]).agg(tv_median=("tv", "median"), null_median=("null_mean", "median"),
                                      sig=("p", lambda p: int((p < 0.05).sum())))
print("\nper class and FT seed (sig = snapshots of 5 with p < 0.05):")
print(summ.round(3).to_string())

  run ft       cls  n_pool  n_test    tv  null_mean     p
r0_s4 r0      silk     420     124 0.321      0.068 0.001
r0_s4 r0 inclusion     135      43 0.113      0.097 0.354
r0_s4 r0    rolled      17       6 0.059      0.085 1.000
r0_s4 r0    crease      32      11 0.270      0.149 0.125
r0_s4 r0     waist      94      30 0.126      0.066 0.147

per class and FT seed (sig = snapshots of 5 with p < 0.05):
              tv_median  null_median  sig
cls       ft                             
crease    r0      0.210        0.143    0
          r1      0.253        0.196    0
          r2      0.156        0.178    0
inclusion r0      0.143        0.109    0
          r1      0.330        0.101    4
          r2      0.118        0.098    0
rolled    r0      0.118        0.150    0
          r1      0.118        0.172    0
          r2      0.049        0.138    0
silk      r0      0.321        0.058    5
          r1      0.186        0.066    5
          r2      0.334        0.066    5
wai

- **Kết quả** (TV trung vị theo FT seed r0 / r1 / r2; số ảnh chụp có p < 0.05 trên 5):
  - silk: 0.321 / 0.186 / 0.334, so với mức nền khoảng 0.06; vượt nền ở 15/15 ảnh chụp. r0 s4: TV 0.321, p = 0.001.
  - inclusion: 0.143 / 0.330 / 0.118; vượt nền ở 4/5 ảnh chụp của r1, 0/5 ở r0 và r2.
  - waist: 0.106 / 0.033 / 0.067; vượt nền 1/15 ảnh chụp.
  - rolled 0/15 và crease 0/15 ảnh chụp vượt nền; mức nền của chúng 0.14–0.20 vì chỉ có 6 và 11 ảnh test.
- **Ý nghĩa**:
  - Dự đoán đúng ở silk (dù r1 thấp hơn khoảng dự đoán: 0.186), ở waist và ở rolled, crease. Sai ở inclusion: chỉ r1 vượt nền.
  - Cách k-means chia silk trên pool không giữ nguyên trên test ở mọi ảnh chụp: Q10.03 đúng với silk.
  - [SUY LUẬN] inclusion ở r1 chuyển cụm mạnh trên test (TV 0.33). Có thể đây là một phần lý do cụm do crease sở hữu ở r1 s0–s2 mang tên inclusion trên test (1.2). Khi đó, ở r1, phần mất của crease có hai nguồn cùng lúc: dịch chuyển pool → test và cách cân theo số ảnh của Hungarian (3.2).
  - Giới hạn vẫn còn: phần TV do tâm cụm học từ pool chưa tách được; 5.2 kiểm xem TV có đi theo cuộn không.

### 5.2 Cụm co lại trên test có phải cụm bám ít cuộn không

Gắn mỗi ảnh pool với cuộn thép của nó: so byte của từng hàng `pool_x` với file đặc trưng train của ảnh chụp, rồi đọc cột `coil` của `data_io.images` (cách làm như `09d` §1.1; không đọc nhãn mới nào).

Với mỗi ô (ảnh chụp, lớp c ∈ {silk, inclusion}, cụm k) có ít nhất 10 ảnh pool:
- độ co Δ = q_c(k) − p_c(k): phần ảnh test trừ phần ảnh pool của lớp c trong cụm k (5.1);
- số cuộn hiệu dụng N_eff = 1/Σ_j s_j², với s_j là tỷ lệ ảnh pool của lớp c trong cụm k thuộc cuộn j (Simpson 1949). N_eff nhỏ nghĩa là ô dồn vào ít cuộn.

Tương quan hạng Spearman giữa N_eff và Δ trên các ô, tính riêng mỗi FT seed và mỗi lớp.

Dự đoán [SUY LUẬN]:
- silk: Spearman dương ở cả 3 FT seed, khoảng 0.4–0.7: ô dồn vào ít cuộn co lại trên test, vì cuộn của test không có trong pool;
- inclusion: yếu hơn, chỉ rõ ở r1, nơi 5.1 thấy dịch chuyển.

In [9]:
# 5.2 coil concentration of each (class, cluster) cell in the pool vs its shrink on the test set
import data_io
from scipy.stats import spearmanr

img = data_io.images("gc10")
tr = img[img["part"] == "train"].set_index("row")
rows52 = []
for run in RUNS:
    b = replay.snapshots(run)
    a = b["04"]["meta"]["args"]; m = a["pretrained_model_name"]
    X = np.load(Path(a["data_dir"]) / m / f"features-{m}.npy")
    where = {x.tobytes(): i for i, x in enumerate(X)}
    rows = np.array([where[x.tobytes()] for x in np.asarray(b["04"]["out"]["pool_x"])])   # pool row -> feature row
    coil = tr.loc[rows, "coil"].to_numpy()
    L = LEDGER[run]
    for c in (5, 6):                                                    # silk, inclusion
        sel = (L["pool"].y == c).to_numpy()
        cp, cc = L["pool"]["cluster"].to_numpy()[sel], coil[sel]
        ct = L["test1"].loc[L["test1"].y == c, "cluster"].to_numpy()
        for k in range(5):
            if (cp == k).sum() < 10:
                continue
            _, cnt = np.unique(cc[cp == k], return_counts=True); s = cnt / cnt.sum()
            rows52.append(dict(run=run.replace("gc10_v2_ft_", ""), ft=run[11:13], cls=names[c], k=k,
                               n_pool=int((cp == k).sum()), p=(cp == k).mean(), q=(ct == k).mean(),
                               n_eff=1 / (s ** 2).sum(), top_coil=s.max()))
T52 = pd.DataFrame(rows52)
T52["shrink"] = T52.q - T52.p
print("rows matched to the feature file: all runs ok")
print(T52[T52.run == "r0_s4"].round(3).to_string(index=False))
print("\nSpearman(n_eff, q - p) per class and FT seed:")
for (cl, ft), g in T52.groupby(["cls", "ft"]):
    r, pv = spearmanr(g.n_eff, g.shrink)
    print(f"  {cl:9s} {ft}: rho {r:+.2f} (p {pv:.3f}, {len(g)} cells)")

rows matched to the feature file: all runs ok
  run ft       cls  k  n_pool     p     q  n_eff  top_coil  shrink
r0_s4 r0      silk  0      29 0.069 0.032  3.058     0.483  -0.037
r0_s4 r0      silk  2     169 0.402 0.694  3.293     0.497   0.291
r0_s4 r0      silk  3     118 0.281 0.169  1.052     0.975  -0.112
r0_s4 r0      silk  4     103 0.245 0.073  1.000     1.000  -0.173
r0_s4 r0 inclusion  0      46 0.341 0.349  9.281     0.217   0.008
r0_s4 r0 inclusion  1      11 0.081 0.186  6.368     0.273   0.105
r0_s4 r0 inclusion  2      76 0.563 0.465 19.253     0.118  -0.098

Spearman(n_eff, q - p) per class and FT seed:
  inclusion r0: rho -0.75 (p 0.005, 12 cells)
  inclusion r1: rho -0.83 (p 0.000, 13 cells)
  inclusion r2: rho -0.46 (p 0.176, 10 cells)
  silk      r0: rho +0.77 (p 0.000, 17 cells)
  silk      r1: rho +0.45 (p 0.041, 21 cells)
  silk      r2: rho +0.47 (p 0.038, 20 cells)


- **Kết quả**:
  - Mọi hàng `pool_x` của 15 ảnh chụp đều tìm được ảnh gốc theo byte.
  - r0 s4, silk: hai cụm co lại trên test là cụm 3 (p 0.281 → q 0.169) và cụm 4 (0.245 → 0.073); cả hai gần như chỉ từ một cuộn (N_eff 1.05 và 1.00; cuộn lớn nhất chiếm 0.975 và 1.000). Cụm 2 trải trên nhiều cuộn (N_eff 3.29) thì lớn lên (0.402 → 0.694).
  - Spearman(N_eff, q − p), r0 / r1 / r2:
    - silk: +0.77 / +0.45 / +0.47 (p ≤ 0.041; 17–21 ô);
    - inclusion: −0.75 / −0.83 / −0.46 (r2: p = 0.18; 10–13 ô).
- **Ý nghĩa**:
  - Dự đoán đúng với silk: ở cả 3 FT seed, ô silk càng dồn vào ít cuộn càng co lại trên test. Ở r0 s4, cả hai cụm silk thừa mang tên rolled và crease đều là cụm của một cuộn duy nhất. Q10.03 đúng với silk.
  - Sai với inclusion: dấu ngược lại, ô của inclusion trải trên nhiều cuộn lại co. [SUY LUẬN] Dịch chuyển của inclusion không do cuộn. Các ô nhiều cuộn của inclusion là phần nằm trong cụm silk lớn (r0 s4: cụm 2, N_eff 19.3), và ảnh inclusion của test rời các cụm đó sang cụm của waist hoặc của chính nó.
  - Giới hạn: các ô trong một ảnh chụp không độc lập (tỷ lệ cộng lại bằng 1), nên p ở đây chỉ để tham khảo; kết luận dựa vào dấu đồng nhất ở 3/3 FT seed.

## Summary (EN)

1. **Step 10 replays exactly.** `rerun` reproduces both keys at r0 s4 and `pi_test` is recomputed identically in 15/15 snapshots. The names come from a count-optimal Hungarian on the k-means cluster of each of the 214 novel test images (`predict_raw`, raw 768-d), not from the Gaussian classifier (§1.1–2.1).
2. **One name per cluster.** Names differ from the pool owner in 1–4 of 5 clusters per snapshot. At r0 s4 two near-pure silk clusters carry the names rolled and crease and score 0/6 and 0/11, while 6/6 rolled and 7/11 crease test images sit in the waist cluster. In r1 s0–s2 the cluster owned by crease in the pool is named inclusion on the test set (§1.2, §2.1).
3. **Count-weighted naming vs a class-weighted metric.** Cluster-level balanced novel under the pipeline naming is 0.458 / 0.386 / 0.405 (r0 / r1 / r2). The recall-optimal naming reaches 0.499 / 0.483 / 0.423 but is inflated by 2.0–4.4 points when chosen on the scored set; cross-fitted, it gains +2.2 / +6.7 / −2.4 (§3.1–3.2). [inference] The pipeline naming stays the main metric's naming (QD15, QD25); the r1 gain is a candidate part of Q14.04, to be checked with the classifier at step 14.
4. **Loss ledger, columns 09 and 10.** `score.ledger` reproduces §3.1 in 15/15 snapshots. At r0 s4 the whole 09 → 10 loss is silk: 30 of 124 test images and 221 of 420 pool images (§4.1).
5. **Q10.03: silk's split does not carry over to the test set.** Within silk, the distribution over clusters differs between pool and test in 15/15 snapshots (total variation 0.19–0.33 against a permutation null of about 0.06), and silk cells drawn from few coils shrink on the test set (Spearman of the effective number of coils with the shrink +0.45 to +0.77 per FT seed). At r0 s4 both silk clusters named rolled and crease come from a single coil. inclusion shifts only in r1, and not along coils (§5.1–5.2).